# Part 3: Fine-tuning Transformers

<div dir="rtl">

این نوت‌بوک دومین نوت‌بوک بخش سوم است: fine-tune کردن مدل‌های ترنسفورمر از پیش‌آموخته برای پیش‌بینی امتیاز ۱ تا ۵. چهار مدل با اندازه‌های مختلف روی داده‌ی یکسان و validation یکسان با نوت‌بوک part3_sentiment آموزش می‌بینند: all-MiniLM-L6-v2 (۲۲ میلیون وزن)، DistilRoBERTa-base (۸۲ میلیون)، RoBERTa-base (۱۲۵ میلیون) و DeBERTa-v3-base (۱۸۴ میلیون). اول چند تنظیم (طول متن، نرخ یادگیری و تعداد epoch) روی مدل کوچک و ۱۰۰ هزار نظر سنجیده می‌شود و بعد هر چهار مدل روی ۲۵۰ هزار نظر آموزش می‌بینند.

نوت‌بوک روی GPU اجرا می‌شود (در کولب T4). فایل‌های لازم در پوشه‌ی project2 در Google Drive: part3_train.parquet و part3_test.parquet، که part3_sentiment.ipynb ساخته است، و part3_tfidf_probs.parquet برای مقایسه. هر مدل در پوشه‌ی finetune ذخیره می‌شود: بهترین وزن‌ها (best.pt)، تاریخچه‌ی آموزش (history.csv) و احتمال‌های پیش‌بینی‌شده برای validation، holdout و test (probs.parquet). وضعیت آموزش همراه هر ارزیابی (چهار بار در هر epoch) در Drive ذخیره می‌شود و اگر کولب قطع شود، اجرای دوباره از همان‌جا ادامه می‌دهد.

</div>

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import time
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import torch
import transformers
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

transformers.logging.set_verbosity_error()
pio.templates.default = 'plotly_white'
path = '/content/drive/MyDrive/project2/'
os.makedirs(path + 'finetune', exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')

device: cuda Tesla T4


## Data

<div dir="rtl">

همان داده‌ی پاک‌شده و همان تقسیم نوت‌بوک قبل خوانده شد. نمونه‌های آموزش با همان random_state ساخته شدند، پس ترنسفورمرها دقیقاً روی همان ۱۰۰ هزار و ۲۵۰ هزار نظری آموزش می‌بینند که مدل‌های TF-IDF دیدند.

</div>

In [3]:
data = pd.read_parquet(path + 'part3_train.parquet')
test = pd.read_parquet(path + 'part3_test.parquet')

fit_pool = data[data['set'] == 'train']
valid = data[data['set'] == 'valid']
holdout = data[data['set'] == 'holdout']
small = fit_pool.groupby('overall').sample(20000, random_state=0)
large = fit_pool.groupby('overall').sample(50000, random_state=0)
print('small:', len(small), ' large:', len(large), ' valid:', len(valid), ' holdout:', len(holdout), ' test:', len(test))

small: 100000  large: 250000  valid: 10000  holdout: 10000  test: 20000


## Model

<div dir="rtl">

مدل دو بخش دارد:

- **encoder:** ترنسفورمر از پیش‌آموخته. متن به tokenها شکسته می‌شود و هر token یک بردار می‌گیرد. در هر لایه، self-attention به هر token اجازه می‌دهد همه‌ی tokenهای دیگر را ببیند و بردارش را بر اساس آن‌ها به‌روز کند؛ بعد یک شبکه‌ی دولایه (feed-forward) روی هر token جداگانه اعمال می‌شود. خروجی هر لایه با ورودی‌اش جمع می‌شود (residual) و نرمال می‌شود (LayerNorm). خروجی آخر برای هر token یک بردار است که معنی آن token را در این جمله نشان می‌دهد.
- **لایه‌ی جدید:** میانگین بردارهای همه‌ی tokenها (بدون padding) یک بردار برای کل نظر می‌سازد و یک لایه‌ی خطی آن را به ۵ عدد، یکی برای هر امتیاز، تبدیل می‌کند. این لایه از صفر آموزش می‌بیند.

در fine-tune کردن، وزن‌های encoder هم همراه لایه‌ی جدید به‌روز می‌شوند، با نرخ یادگیری کوچک تا دانش از پیش‌آموخته از بین نرود. این همان ساختار برای هر سه مدل است تا مقایسه منصفانه باشد.

وزن‌ها همیشه با دقت ۳۲ بیتی بارگذاری می‌شوند. transformers نسخه‌ی ۵ وزن‌ها را با همان دقتی بارگذاری می‌کند که در فایل ذخیره شده‌اند و فایل DeBERTa-v3 با ۱۶ بیت ذخیره شده است؛ به‌روز کردن وزن‌های ۱۶ بیتی دقیق نیست. بخش‌هایی از checkpointها که در این مدل استفاده نمی‌شوند (lm_head مخصوص پیش‌آموزش و pooler) کنار می‌مانند و پیام‌های بارگذاری درباره‌ی آن‌ها نمایش داده نمی‌شود.

</div>

In [4]:
class Classifier(torch.nn.Module):
    def __init__(self, name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name, dtype=torch.float32)
        self.dropout = torch.nn.Dropout(0.1)
        self.head = torch.nn.Linear(self.encoder.config.hidden_size, 5)

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        return self.head(self.dropout(pooled))

<div dir="rtl">

summary و متن نظر به‌صورت یک جفت به tokenizer داده می‌شوند. tokenizer بینشان token جداکننده می‌گذارد و اگر طول از حد بیشتر شد فقط متن نظر کوتاه می‌شود. در حالت head+tail، نظرهای بلند قبل از tokenize شدن از وسط کوتاه می‌شوند، چون نویسنده‌ها جمع‌بندی‌شان را معمولاً آخر نظر می‌نویسند.

</div>

In [5]:
def head_tail(text, head=700, tail=400):
    if len(text) <= head + tail:
        return text
    return text[:head] + ' ... ' + text[-tail:]

def encode(tokenizer, df, max_length, trim, chunk=20000):
    summaries = df['summary'].str[:200].tolist()
    reviews = (df['reviewText'].map(head_tail) if trim else df['reviewText']).tolist()
    ids = []
    # in chunks, so the tokenizer's intermediate objects for all reviews are never in RAM at once
    for start in range(0, len(reviews), chunk):
        ids += tokenizer(summaries[start:start + chunk], reviews[start:start + chunk], truncation='only_second',
                         max_length=max_length)['input_ids']
    return ids

def epoch_batches(lengths, batch_size, epoch):
    rng = np.random.default_rng(epoch)
    order = rng.permutation(len(lengths))
    batches = []
    for start in range(0, len(order), batch_size * 50):
        chunk = order[start:start + batch_size * 50]
        chunk = chunk[np.argsort(lengths[chunk], kind='stable')]
        batches += [chunk[i:i + batch_size] for i in range(0, len(chunk), batch_size)]
    return [batches[i] for i in rng.permutation(len(batches))]

def make_batch(ids_list, pad_id):
    length = max(len(ids) for ids in ids_list)
    input_ids = torch.full((len(ids_list), length), pad_id)
    mask = torch.zeros((len(ids_list), length), dtype=torch.long)
    for row, ids in enumerate(ids_list):
        input_ids[row, :len(ids)] = torch.tensor(ids)
        mask[row, :len(ids)] = 1
    return input_ids.to(device), mask.to(device)

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
example = encode(tokenizer, valid.head(1), 256, False)[0]
print(tokenizer.decode(example)[:300])

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[CLS] heavy, secure. and not very portable. [SEP] this is a very nice, secure way to lock up your macbook, now that they don ' t come with a keyhole. it is easy to put together - instructions are simple. the only tool you need is included. very straightforward. heavy duty. no one will cut this cable


## Training loop

<div dir="rtl">

هر قدم آموزش:

1. یک دسته (batch) از نظرها برداشته می‌شود (۳۲ نظر برای MiniLM و ۱۶ نظر برای سه مدل دیگر) و با padding هم‌طول می‌شود. نظرها اول به‌طور تصادفی به گروه‌های ۵۰ دسته‌ای تقسیم می‌شوند و داخل هر گروه بر اساس طول مرتب می‌شوند، بعد دسته‌ها با ترتیب تصادفی آموزش داده می‌شوند. این‌طور نظرهای هم‌طول در یک دسته می‌افتند و padding خیلی کمتر می‌شود، ولی ترتیب آموزش همچنان تصادفی است.
2. مدل برای هر نظر ۵ عدد می‌دهد و cross-entropy فاصله‌ی آن‌ها را با امتیاز واقعی می‌سنجد.
3. گرادیان با backpropagation حساب می‌شود و اندازه‌اش حداکثر ۱ نگه داشته می‌شود تا یک دسته‌ی غیرعادی مدل را به‌هم نریزد.
4. AdamW وزن‌ها را به‌روز می‌کند. weight decay وزن‌ها را کمی به سمت صفر می‌کشد (regularization).
5. نرخ یادگیری در ۶٪ اول قدم‌ها از صفر بالا می‌رود (warmup) تا لایه‌ی تصادفی جدید در شروع کار encoder را خراب نکند، و بعد تا صفر پایین می‌آید.

محاسبه با دقت ۱۶ بیتی (fp16) انجام می‌شود که روی GPU سریع‌تر است و حافظه‌ی کمتری می‌گیرد. GradScaler loss را قبل از backpropagation بزرگ می‌کند تا گرادیان‌های کوچک در ۱۶ بیت صفر نشوند. چند بار در هر epoch درستی روی validation حساب می‌شود و بهترین وزن‌ها نگه داشته می‌شوند. ترتیب نظرها در هر epoch با seed ثابت است، پس اگر آموزش قطع شود، از همان قدم با همان ترتیب ادامه پیدا می‌کند. اگر یک دسته در حافظه‌ی GPU جا نشود، می‌شود آن را به چند micro-batch تقسیم کرد (پارامتر micro_batch)؛ این فقط برای DeBERTa لازم شد و پایین‌تر توضیح داده شده.

هشدار «lr_scheduler.step() before optimizer.step()» در خروجی DistilRoBERTa از GradScaler می‌آید. اگر در قدم‌های اول گرادیان در ۱۶ بیت بی‌نهایت شود، GradScaler آن قدم را رد می‌کند و ضریب بزرگ‌نمایی loss را کم می‌کند، ولی scheduler همان یک قدم را جلو می‌رود. اثرش فقط جابه‌جا شدن برنامه‌ی نرخ یادگیری به اندازه‌ی همان چند قدم است، از حدود ۳۱ هزار قدم آموزش.

</div>

In [6]:
@torch.no_grad()
def predict(model, ids, pad_id, batch_size=32):
    model.eval()
    order = np.argsort([len(x) for x in ids])
    probs = np.zeros((len(ids), 5), dtype=np.float32)
    for start in range(0, len(ids), batch_size):
        idx = order[start:start + batch_size]
        input_ids, mask = make_batch([ids[i] for i in idx], pad_id)
        with torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
            logits = model(input_ids, mask)
        probs[idx] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
    model.train()
    return probs

In [7]:
def train(run, name, fit, max_length=256, trim=False, lr=2e-5, epochs=2, batch_size=32, micro_batch=None,
          evals_per_epoch=4):
    folder = path + 'finetune/' + run + '/'
    if os.path.exists(folder + 'probs.parquet'):
        print(run, 'is already trained. Training log:')
        print(pd.read_csv(folder + 'history.csv').to_string(index=False))
        return
    os.makedirs(folder, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(name)
    pad_id = tokenizer.pad_token_id
    fit_ids = encode(tokenizer, fit, max_length, trim)
    fit_lengths = np.array([len(ids) for ids in fit_ids])
    valid_ids = encode(tokenizer, valid, max_length, trim)
    labels = torch.tensor(fit['overall'].values - 1)

    torch.manual_seed(0)
    model = Classifier(name).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps_per_epoch = len(epoch_batches(fit_lengths, batch_size, 0))
    total_steps = steps_per_epoch * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)
    scaler = torch.amp.GradScaler(enabled=device == 'cuda')
    eval_every = steps_per_epoch // evals_per_epoch
    micro_batch = micro_batch or batch_size

    step, history, best, seconds = 0, [], 0, 0
    resumed = os.path.exists(folder + 'checkpoint.pt')
    if resumed:
        state = torch.load(folder + 'checkpoint.pt', map_location=device, weights_only=False)
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        step, history, best, seconds = state['step'], state['history'], state['best'], state['seconds']
        del state
        print('resumed at step', step)

    model.train()
    losses, start = [], time.time()
    while step < total_steps:
        epoch, position = divmod(step, steps_per_epoch)
        batches = epoch_batches(fit_lengths, batch_size, epoch)
        for position in range(position, steps_per_epoch):
            idx = batches[position]
            optimizer.zero_grad()
            batch_loss = 0.0
            for part in range(0, len(idx), micro_batch):
                part_idx = idx[part:part + micro_batch]
                input_ids, mask = make_batch([fit_ids[i] for i in part_idx], pad_id)
                with torch.autocast(device, dtype=torch.float16, enabled=device == 'cuda'):
                    logits = model(input_ids, mask)
                # weighted by share of the batch, so summed gradients equal the full-batch gradient
                loss = torch.nn.functional.cross_entropy(logits.float(), labels[torch.as_tensor(part_idx)].to(device))
                loss = loss * len(part_idx) / len(idx)
                if not torch.isfinite(loss):
                    raise ValueError(f'loss is {loss.item()} at step {step}')
                scaler.scale(loss).backward()
                batch_loss += loss.item()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            losses.append(batch_loss)
            step += 1

            if step % eval_every == 0 or step == total_steps:
                accuracy = (predict(model, valid_ids, pad_id).argmax(1) + 1 == valid['overall'].values).mean()
                seconds += time.time() - start
                start = time.time()
                history.append({'step': step, 'epoch': round(step / steps_per_epoch, 2), 'train_loss': np.mean(losses),
                                'valid_accuracy': accuracy, 'minutes': round(seconds / 60, 1)})
                print(f"epoch {history[-1]['epoch']:.2f}  train loss {np.mean(losses):.4f}  "
                      f"valid accuracy {accuracy:.4f}  ({seconds / 60:.0f} min)", flush=True)
                losses = []
                if accuracy > best:
                    best = accuracy
                    torch.save(model.state_dict(), folder + 'best.pt')
                torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                            'scheduler': scheduler.state_dict(), 'scaler': scaler.state_dict(),
                            'step': step, 'history': history, 'best': best, 'seconds': seconds},
                           folder + 'checkpoint.pt')
            if step >= total_steps:
                break

    pd.DataFrame(history).to_csv(folder + 'history.csv', index=False)
    model.load_state_dict(torch.load(folder + 'best.pt'))
    probs = []
    for name_, df in [('valid', valid), ('holdout', holdout), ('test', test)]:
        p = predict(model, encode(tokenizer, df, max_length, trim), pad_id)
        probs.append(pd.DataFrame(p, columns=['p1', 'p2', 'p3', 'p4', 'p5'], index=df.index).assign(set=name_))
    pd.concat(probs).to_parquet(folder + 'probs.parquet')
    os.remove(folder + 'checkpoint.pt')
    if resumed:
        print('full training log:')
        print(pd.DataFrame(history).to_string(index=False))
    print(run, 'best valid accuracy:', round(best, 4))

<div dir="rtl">

وقتی آموزش تمام می‌شود، بهترین وزن‌ها دوباره بارگذاری می‌شوند و احتمال‌ها برای validation، holdout و test ذخیره می‌شوند. درستی روی holdout اینجا حساب نمی‌شود. فایل checkpoint (وزن‌ها به‌علاوه‌ی وضعیت optimizer) فقط برای ادامه دادن بعد از قطع شدن لازم است و آخر کار پاک می‌شود.

</div>

## Settings on the small model

<div dir="rtl">

سه سؤال با مدل کوچک (all-MiniLM-L6-v2) و ۱۰۰ هزار نظر بررسی شد، چون هر اجرا فقط چند دقیقه طول می‌کشد:

- **طول متن:** بخشی از نظر که از حد مجاز بلندتر باشد کنار می‌رود. ۲۵۶ token بیشتر نظرها را کامل جا می‌دهد و ۵۱۲ token تقریباً همه را، ولی آموزش با ۵۱۲ کندتر است (سهم دقیق هر طول در part3_final.ipynb حساب شده). حالت سوم ۲۵۶ token با head+tail است.
- **نرخ یادگیری:** مدل‌های کوچک معمولاً نرخ بزرگ‌تری لازم دارند. ۵e-5 و ۱e-4 مقایسه شدند.
- **تعداد epoch:** هر اجرا دو epoch دارد و درستی چهار بار در هر epoch حساب می‌شود؛ منحنی نشان می‌دهد epoch دوم کمک می‌کند یا نه.

</div>

In [8]:
minilm = 'sentence-transformers/all-MiniLM-L6-v2'
train('minilm_small_256', minilm, small, max_length=256, lr=5e-5)

minilm_small_256 is already trained. Training log:
 step  epoch  train_loss  valid_accuracy  minutes
  781   0.25    1.147613          0.5585      0.7
 1562   0.50    0.950318          0.6215      1.5
 2343   0.75    0.906663          0.6238      2.3
 3124   1.00    0.867149          0.6279      3.2
 3905   1.25    0.799837          0.6401      4.0
 4686   1.50    0.795822          0.6397      4.8
 5467   1.75    0.778019          0.6442      5.7
 6248   2.00    0.777211          0.6480      6.6
 6250   2.00    0.877670          0.6482      6.9


In [9]:
train('minilm_small_256_headtail', minilm, small, max_length=256, trim=True, lr=5e-5)

minilm_small_256_headtail is already trained. Training log:
 step  epoch  train_loss  valid_accuracy  minutes
  781   0.25    1.145149          0.5688      0.7
 1562   0.50    0.944514          0.6181      1.6
 2343   0.75    0.899025          0.6227      2.4
 3124   1.00    0.864397          0.6259      3.3
 3905   1.25    0.795201          0.6381      4.2
 4686   1.50    0.790486          0.6434      5.0
 5467   1.75    0.768880          0.6444      5.9
 6248   2.00    0.768146          0.6493      6.8
 6250   2.00    0.904966          0.6496      7.0


In [10]:
train('minilm_small_512', minilm, small, max_length=512, lr=5e-5)

minilm_small_512 is already trained. Training log:
 step  epoch  train_loss  valid_accuracy  minutes
  781   0.25    1.139083          0.5763      0.9
 1562   0.50    0.947170          0.6196      1.9
 2343   0.75    0.893289          0.6253      3.0
 3124   1.00    0.861314          0.6330      3.9
 3905   1.25    0.790655          0.6470      4.9
 4686   1.50    0.780482          0.6526      5.9
 5467   1.75    0.763208          0.6554      7.0
 6248   2.00    0.767760          0.6554      8.1
 6250   2.00    0.965179          0.6555      8.2


In [11]:
train('minilm_small_256_lr1e-4', minilm, small, max_length=256, lr=1e-4)

minilm_small_256_lr1e-4 is already trained. Training log:
 step  epoch  train_loss  valid_accuracy  minutes
  781   0.25    1.122124          0.5978      0.7
 1562   0.50    0.944542          0.6203      1.5
 2343   0.75    0.905772          0.6259      2.4
 3124   1.00    0.866346          0.6240      3.5
 3905   1.25    0.777535          0.6379      4.3
 4686   1.50    0.776631          0.6406      5.2
 5467   1.75    0.752549          0.6462      6.1
 6248   2.00    0.751149          0.6486      6.9
 6250   2.00    0.867926          0.6488      7.0


In [12]:
def history_of(runs):
    return pd.concat([pd.read_csv(path + 'finetune/' + r + '/history.csv').assign(run=r) for r in runs])

settings_runs = ['minilm_small_256', 'minilm_small_256_headtail', 'minilm_small_512', 'minilm_small_256_lr1e-4']
settings = history_of(settings_runs)
fig = px.line(settings, x='epoch', y='valid_accuracy', color='run', markers=True,
              title='MiniLM on 100,000 reviews: validation accuracy during training')
fig.show()
settings.groupby('run').agg(best_accuracy=('valid_accuracy', 'max'), minutes=('minutes', 'max')).round(4)

,best_accuracy,minutes
run,,
minilm_small_256,0.6482,6.9
minilm_small_256_headtail,0.6496,7.0
minilm_small_256_lr1e-4,0.6488,7.0
minilm_small_512,0.6555,8.2


<div dir="rtl">

طول ۵۱۲ بهترین بود: ۰.۶۵۵۵ در برابر ۰.۶۴۸۲ برای ۲۵۶، یعنی ۰.۷ امتیاز بیشتر. با دسته‌های هم‌طول، آموزش با ۵۱۲ فقط حدود ۲۰٪ کندتر بود (۸.۲ در برابر ۶.۹ دقیقه)، چون فقط نظرهای بلند هزینه‌ی بیشتری دارند. head+tail با ۲۵۶ token (۰.۶۴۹۶) تقریباً هم‌سطح کوتاه کردن معمولی است؛ دیدن کل متن بیشتر از انتخاب بخشی از آن کمک می‌کند. نرخ یادگیری ۱e-4 (۰.۶۴۸۸) و ۵e-5 (۰.۶۴۸۲) عملاً برابرند. در هر چهار اجرا epoch دوم ۲ تا ۲.۵ امتیاز اضافه کرد، پس دو epoch لازم است.

</div>

<div dir="rtl">

تنظیم‌های مدل‌های بزرگ‌تر با این قاعده انتخاب می‌شوند که پیش از دیدن نتیجه‌ها نوشته شد: ۵۱۲ token فقط اگر دست‌کم ۰.۵ امتیاز بهتر از بهترین حالت ۲۵۶ باشد، چون آموزش را تقریباً دو برابر طول می‌دهد؛ head+tail اگر از کوتاه کردن معمولی بهتر باشد؛ و برای MiniLM نرخ یادگیری بهتر. نرخ یادگیری RoBERTa و DeBERTa روی مدل کوچک تنظیم نمی‌شود و ۲e-5 است، که در بازه‌ی نرخ‌های fine-tune گزارش‌شده در مقاله‌های این دو مدل است.

</div>

In [13]:
best_runs = settings.groupby('run')['valid_accuracy'].max()
best_256 = max(best_runs['minilm_small_256'], best_runs['minilm_small_256_headtail'])
max_length = 512 if best_runs['minilm_small_512'] > best_256 + 0.005 else 256
trim = max_length == 256 and best_runs['minilm_small_256_headtail'] > best_runs['minilm_small_256']
minilm_lr = 1e-4 if best_runs['minilm_small_256_lr1e-4'] > best_runs['minilm_small_256'] else 5e-5
base_batch = 32 if max_length == 256 else 16
print('max_length:', max_length, ' head+tail:', trim, ' MiniLM learning rate:', minilm_lr, ' batch size for base models:', base_batch)

max_length: 512  head+tail: False  MiniLM learning rate: 0.0001  batch size for base models: 16


<div dir="rtl">

قاعده ۵۱۲ token را انتخاب کرد: ۰.۶ امتیاز بهتر از بهترین حالت ۲۵۶ (head+tail با ۰.۶۴۹۶)، کمی بالاتر از آستانه‌ی ۰.۵. با ۵۱۲ token، head+tail لازم نیست و کنار رفت. برای MiniLM نرخ ۱e-4 انتخاب شد که فاصله‌اش با ۵e-5 فقط ۰.۰۶ امتیاز است. اندازه‌ی دسته‌ی DistilRoBERTa، RoBERTa و DeBERTa ۱۶ است تا با ۵۱۲ token در حافظه‌ی GPU جا شوند.

</div>

## Four models on 250,000 reviews

<div dir="rtl">

هر چهار مدل با تنظیم‌های بالا، دو epoch روی همان ۲۵۰ هزار نظر متوازن مدل نهایی TF-IDF آموزش می‌بینند. RoBERTa-base همان معماری BERT با ۱۲ لایه است که با داده‌ی خیلی بیشتر و بدون هدف پیش‌بینی جمله‌ی بعد پیش‌آموزش دیده. DeBERTa-v3-base هم ۱۲ لایه دارد ولی در attention محتوای هر token و فاصله‌ی نسبی tokenها را جدا حساب می‌کند و در پیش‌آموزش یاد گرفته تشخیص بدهد کدام tokenها جایگزین شده‌اند. MiniLM نسخه‌ی ۶ لایه و کوچک‌شده‌ای از همین خانواده است.

DistilRoBERTa-base نسخه‌ی ۶ لایه‌ی RoBERTa-base است که با distillation ساخته شده: مدل کوچک یاد گرفته خروجی RoBERTa را تقلید کند. tokenizer و اندازه‌ی بردارها (۷۶۸) همان RoBERTa است، پس فاصله‌ی این دو مدل اثر نصف کردن لایه‌ها را نشان می‌دهد. این مدل را یکی از اعضای گروه جدا از این نوت‌بوک، با نمونه و validation دیگری، آموزش داده بود. اینجا با همان داده و همان تنظیم‌های RoBERTa (نرخ یادگیری ۲e-5، دسته‌ی ۱۶تایی، ۵۱۲ token) دوباره آموزش می‌بیند تا با بقیه قابل مقایسه باشد.

آموزش DeBERTa در اجرای اول بعد از epoch اول قطع شد، چون سهمیه‌ی GPU رایگان کولب تمام شد. اجرای نهایی از checkpoint پایان epoch اول (قدم ۱۵۶۲۴) ادامه داد، و ستون minutes در تاریخچه‌ی DeBERTa زمان هر دو اجرا را با هم نشان می‌دهد.

DeBERTa در حافظه‌ی T4 (۱۵ گیگابایت) لب مرز است. RoBERTa و MiniLM از kernel ادغام‌شده‌ی scaled dot-product attention استفاده می‌کنند که ماتریس کامل attention را در حافظه نگه نمی‌دارد. disentangled attention در DeBERTa این kernel را ندارد: در هر لایه و برای هر head، ماتریس‌های ۵۱۲×۵۱۲ امتیاز content-to-content، content-to-position و position-to-content کامل ساخته می‌شوند و خروجی softmax برای backpropagation نگه داشته می‌شود. epoch اول با دسته‌های ۱۶تایی کامل آموزش دید، ولی اولین تلاش برای ادامه دادن از checkpoint روی یکی از دسته‌های ۵۱۲ tokenی با کمبود حافظه‌ی GPU (out of memory) متوقف شد. برای همین در epoch دوم هر دسته‌ی ۱۶تایی به دو micro-batch هشت‌تایی تقسیم شد: گرادیان هر micro-batch جدا حساب می‌شود، گرادیان‌ها با هم جمع می‌شوند و بعد یک بار وزن‌ها به‌روز می‌شوند (gradient accumulation). loss هر micro-batch به نسبت سهمش از دسته وزن داده می‌شود، پس گرادیان همان گرادیان دسته‌ی ۱۶تایی است و تعداد قدم‌ها و برنامه‌ی نرخ یادگیری با epoch اول و با RoBERTa یکی می‌ماند.

</div>

In [14]:
train('minilm', minilm, large, max_length=max_length, trim=trim, lr=minilm_lr)

minilm is already trained. Training log:
 step  epoch  train_loss  valid_accuracy  minutes
 1953   0.25    1.030602          0.6227      2.0
 3906   0.50    0.873830          0.6500      4.5
 5859   0.75    0.843674          0.6515      7.0
 7812   1.00    0.822593          0.6574      9.7
 9765   1.25    0.750182          0.6631     12.4
11718   1.50    0.735513          0.6572     15.1
13671   1.75    0.733235          0.6696     17.5
15624   2.00    0.719580          0.6723     20.2
15626   2.00    0.785479          0.6723     20.9


In [15]:
train('distilroberta', 'distilbert/distilroberta-base', large, max_length=max_length, trim=trim, lr=2e-5,
      batch_size=base_batch)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_879/436385776.py:65: UserWarning:

Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate



epoch 0.25  train loss 1.0016  valid accuracy 0.6367  (6 min)
epoch 0.50  train loss 0.8379  valid accuracy 0.6595  (13 min)
epoch 0.75  train loss 0.8036  valid accuracy 0.6622  (19 min)
epoch 1.00  train loss 0.7847  valid accuracy 0.6627  (26 min)
epoch 1.25  train loss 0.7231  valid accuracy 0.6695  (32 min)
epoch 1.50  train loss 0.7105  valid accuracy 0.6742  (38 min)
epoch 1.75  train loss 0.7091  valid accuracy 0.6767  (45 min)
epoch 2.00  train loss 0.7002  valid accuracy 0.6789  (52 min)
epoch 2.00  train loss 0.8233  valid accuracy 0.6788  (52 min)
distilroberta best valid accuracy: 0.6789


In [16]:
train('roberta', 'FacebookAI/roberta-base', large, max_length=max_length, trim=trim, lr=2e-5, batch_size=base_batch)

roberta is already trained. Training log:
 step  epoch  train_loss  valid_accuracy  minutes
 3906   0.25    0.946193          0.6566     10.8
 7812   0.50    0.808222          0.6686     24.4
11718   0.75    0.772395          0.6717     37.6
15624   1.00    0.760357          0.6750     51.5
19530   1.25    0.689906          0.6871     64.9
23436   1.50    0.675915          0.6850     78.7
27342   1.75    0.674821          0.6919     91.8
31248   2.00    0.665922          0.6920    105.4
31250   2.00    0.799392          0.6920    108.5


In [17]:
train('deberta', 'microsoft/deberta-v3-base', large, max_length=max_length, trim=trim, lr=2e-5, batch_size=base_batch,
      micro_batch=base_batch // 2)

deberta is already trained. Training log:
 step  epoch  train_loss  valid_accuracy  minutes
 3906   0.25    0.910924          0.6694     21.6
 7812   0.50    0.779971          0.6814     47.4
11718   0.75    0.745235          0.6880     76.9
15624   1.00    0.732339          0.6942    106.9
19530   1.25    0.663206          0.6919    129.8
23436   1.50    0.657593          0.6983    153.6
27342   1.75    0.656476          0.7009    178.1
31248   2.00    0.641739          0.7033    202.2
31250   2.00    0.559140          0.7032    203.9


## Comparison

<div dir="rtl">

هر چهار مدل و مدل TF-IDF نوت‌بوک قبل روی همان validation مقایسه شدند. درستی validation در اینجا از همان بهترین checkpoint است که برای هر مدل انتخاب شد.

</div>

In [18]:
from sklearn.metrics import accuracy_score, f1_score

def scores(probs):
    v = probs[probs['set'] == 'valid']
    pred = v[['p1', 'p2', 'p3', 'p4', 'p5']].values.argmax(1) + 1
    y = data.loc[v.index, 'overall'].values
    return accuracy_score(y, pred), f1_score(y, pred, average='macro')

rows = [('TF-IDF + logistic regression', None, *scores(pd.read_parquet(path + 'part3_tfidf_probs.parquet')), None)]
for run, params in [('minilm', 22.7), ('distilroberta', 82), ('roberta', 125), ('deberta', 184)]:
    folder = path + 'finetune/' + run + '/'
    minutes = pd.read_csv(folder + 'history.csv')['minutes'].max()
    rows.append((run, params, *scores(pd.read_parquet(folder + 'probs.parquet')), minutes))
comparison = pd.DataFrame(rows, columns=['model', 'million parameters', 'accuracy', 'macro_f1', 'training minutes'])
comparison.round(4)

,model,million parameters,accuracy,macro_f1,training minutes
0,TF-IDF + logistic regression,NaN,0.6032,0.6001,NaN
1,minilm,22.7,0.6723,0.6725,20.9
2,distilroberta,82.0,0.6789,0.6786,52.0
3,roberta,125.0,0.6920,0.6920,108.5
4,deberta,184.0,0.7033,0.7030,203.9


In [19]:
fig = px.line(history_of(['minilm', 'distilroberta', 'roberta', 'deberta']), x='epoch', y='valid_accuracy', color='run', markers=True,
              title='Validation accuracy during training on 250,000 reviews')
fig.show()

<div dir="rtl">

DeBERTa-v3-base با ۰.۷۰۳ بهترین مدل است، بعد RoBERTa-base با ۰.۶۹۲، DistilRoBERTa با ۰.۶۷۹ و MiniLM با ۰.۶۷۲. هر چهار ترنسفورمر از مدل TF-IDF (۰.۶۰۳) خیلی بهترند: از ۶.۹ امتیاز برای MiniLM تا ۱۰ امتیاز برای DeBERTa. macro-F1 تقریباً با درستی برابر است، چون validation متوازن است. معیار ارزیابی پروژه micro-F1 است که در دسته‌بندی تک‌برچسبی همان درستی (accuracy) است.

مدل بزرگ‌تر بهتر است، ولی هر قدم سود کمتری دارد و زمان بیشتری می‌گیرد. DistilRoBERTa با ۵۲ دقیقه آموزش ۰.۷ امتیاز از MiniLM (۲۰.۹ دقیقه) بهتر است. RoBERTa با ۱۰۸.۵ دقیقه ۱.۳ امتیاز از DistilRoBERTa بهتر است، و DeBERTa با ۲۰۳.۹ دقیقه ۱.۱ امتیاز از RoBERTa.

DistilRoBERTa و RoBERTa tokenizer و اندازه‌ی بردار یکسان دارند و فقط تعداد لایه‌هایشان فرق دارد (۶ در برابر ۱۲). نصف کردن لایه‌ها زمان آموزش را نصف کرد و ۱.۳ امتیاز درستی کم کرد. RoBERTa و DeBERTa هر دو ۱۲ لایه دارند، پس سود DeBERTa از عمق بیشتر نیست؛ فرق این دو در attention و روش پیش‌آموزش است (بخش قبل).

epoch دوم برای هر چهار مدل مفید بود: MiniLM ۱.۵ امتیاز (۰.۶۵۷ به ۰.۶۷۲)، DistilRoBERTa ۱.۶ امتیاز (۰.۶۶۳ به ۰.۶۷۹)، RoBERTa ۱.۷ امتیاز (۰.۶۷۵ به ۰.۶۹۲) و DeBERTa ۰.۹ امتیاز (۰.۶۹۴ به ۰.۷۰۳). بهترین checkpoint هر چهار مدل در پایان آموزش است. منحنی RoBERTa در ربع آخر صاف شد (۰.۶۹۱۹ به ۰.۶۹۲۰) و DeBERTa هنوز کمی بالا می‌رفت (۰.۷۰۰۹ به ۰.۷۰۳۳). epoch سوم امتحان نشد، چون هر epoch مدل DeBERTa حدود ۱۰۰ دقیقه GPU لازم دارد.

اجرای جداگانه‌ی DistilRoBERTa که یکی از اعضای گروه انجام داده بود (۲۵۶ token با head+tail، نرخ یادگیری ۳e-5 و نمونه‌ی متوازن تازه در هر epoch) با این جدول مقایسه‌پذیر نیست، چون روی validation دیگری سنجیده شده و نظرهای تکراری از آن داده حذف نشده بودند. اجرای این نوت‌بوک همان مدل را روی همان داده و همان validation بقیه می‌سنجد.

انتخاب مدل نهایی، ترکیب مدل‌ها و ارزیابی روی holdout در part3_final.ipynb انجام می‌شود.

</div>